# Phase 9 — Does the AI Earn Its Keep?

**The idea in one line.** The dissertation's claim is not "a VAE can trade" — it is "a VAE finds
*better pairs* than simpler methods". This notebook is the trial: the **identical** walk-forward
machinery from Phase 6 is re-run with the VAE swapped out for two simpler pair-finders, plus a
do-nothing market benchmark, all charged the same trading costs.

> **Change from the approved proposal.** The proposal named PCA as the comparator. On the
> supervisor's guidance this was swapped for **GARCH** — a classical statistics workhorse — as a
> simpler and fairer "does the machine-learning complexity earn its keep?" baseline. Only the
> comparator changed; the structure of the comparison (same pipeline, encoder swapped) is unchanged.

**The four runners:**

1. **VAE chain** (the headline) — Phase 6's result, read straight from its saved files.
2. **GARCH chain** — same clustering, same cointegration tests, same filters, same trading rule,
   same costs; only the fingerprint is different (volatility statistics instead of learned features).
3. **Correlation baseline** — the classical approach this whole project challenges: skip fingerprints
   and clustering entirely, shortlist the most-correlated pairs, then apply the same tests, filters,
   rule, and costs.
4. **Buy-and-hold** — the equal-weight market, for absolute context.

**Why this is a fair trial.** Every runner imports the same `walkforward.py` written by Phase 6 —
the same quarterly calendar, the same five-year formation window, the same stability guard, the same
gates, the same z-score rule, the same cost toll from Phase 8. The only degree of freedom is *how
candidate pairs are found*. Whatever separates the equity curves, it can only be the encoder.

## The GARCH fingerprint, in plain words

**What it is.** GARCH is the classical statistician's tool for describing how a stock's *jumpiness*
behaves: volatility comes in clusters (rough days follow rough days), and GARCH captures that with a
handful of fitted numbers. From each stock's fit we keep four: how **sticky** its jumpiness is (do
storms linger?), its **long-run** jumpiness level (how rough is a normal year?), and the **average**
and **variability** of its fitted day-to-day jumpiness over the window. Four numbers = a "volatility
personality" per stock.

**Why it is the right comparator.** The VAE reads the *shape* of a stock's moves; GARCH reads only
their *jumpiness*. If pairs found by volatility-personality trade just as well as pairs found by the
VAE's learned fingerprints, the deep model adds nothing over a 1980s statistical model — that is
precisely the question the examiner wants answered. And like the frozen VAE re-encoding fresh
windows, GARCH is simply re-fitted on each five-year formation window: for a classical model the fit
*is* the encoding, so both runners get fresh fingerprints from identical data at every rebalance.

*Technical note (safe to skip):* per stock we fit GARCH(1,1) on daily log returns ×100 and keep
[α+β (persistence), ω/(1−α−β) (long-run variance), mean and std of the fitted conditional
volatility]. The occasional fit that fails to converge or sits at the α+β≈1 boundary is recorded,
and its features are filled with that window's cross-sectional median — the count is printed so
nothing is hidden.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from arch import arch_model
from walkforward import cluster_selector, corr_selector, run_walk_forward, perf

tickers = pd.read_csv('data/model_input/tickers.csv', header=None)[0].tolist()
windows = pd.read_csv('data/model_input/window_dates.csv', index_col='window')
prices = pd.read_parquet('data/processed/adj_close_clean.parquet')[tickers]
logp = np.log(prices)
ret = prices.pct_change()
lr100 = logp.diff() * 100                       # per-cent units keep the GARCH optimiser stable

LOOKBACK = 20
garch_fit_log = []

def garch_features(w):
    r = lr100.loc[windows.start_date[w - LOOKBACK]:windows.end_date[w - 1]]
    rows, degenerate = [], 0
    for t in r.columns:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                res = arch_model(r[t].dropna(), vol='GARCH', p=1, q=1).fit(disp='off', show_warning=False)
            a, b, om = res.params['alpha[1]'], res.params['beta[1]'], res.params['omega']
            cv = res.conditional_volatility
            if a + b < 0.999:
                rows.append([a + b, om / (1 - a - b), cv.mean(), cv.std()])
            else:
                rows.append([np.nan] * 4); degenerate += 1
        except Exception:
            rows.append([np.nan] * 4); degenerate += 1
    F = pd.DataFrame(rows)
    F = F.fillna(F.median()).values
    sd = F.std(0); sd[sd == 0] = 1.0
    garch_fit_log.append({'window': w, 'degenerate_fits': degenerate})
    return (F - F.mean(0)) / sd

## Run the GARCH chain

Identical call to Phase 6's, with only the fingerprint function swapped. **Expect this to be the slow
cell — roughly 10–15 minutes** (27 quarters × 462 GARCH fits each, plus the usual cointegration
tests). One line prints per quarter, exactly like Phase 6.

**Red flags to watch:** a large degenerate-fit count (printed at the end — a handful per quarter is
normal, hundreds would mean the volatility features are mush), or the selector starving (0–2 pairs
quarter after quarter).

In [ ]:
g_ret, g_tvr, g_pairs, g_summary = run_walk_forward(
    cluster_selector(garch_features), logp, ret, windows, label='GARCH')

print()
print(pd.DataFrame(garch_fit_log).set_index('window').T.to_string())
print(f'total degenerate fits: {sum(d["degenerate_fits"] for d in garch_fit_log)} '
      f'of {27 * len(tickers)}')

## Run the correlation baseline

The classical selector this project set out to beat: at each rebalance, compute the correlation of
daily returns between every pair of stocks in the whole 462-stock universe (one cheap matrix
operation), take the **200 most-correlated pairs**, and hand them to the *same* cointegration tests,
gates, and trading rule.

**Why 200?** The cluster-based chains test a few hundred within-group pairs per rebalance; capping
the correlation shortlist at the same order of magnitude gives every selector a comparable
"fluke budget" under the p < 0.05 test (test vastly more pairs and you harvest more lucky ones — an
unfair advantage that has nothing to do with selection skill).

In [ ]:
c_ret, c_tvr, c_pairs, c_summary = run_walk_forward(
    corr_selector(top_n=200), logp, ret, windows, label='CORR')

## Buy-and-hold, and one toll booth for everyone

Buy-and-hold is the "why bother?" benchmark: put the money in the equal-weight market on day one and
sit still. It pays the toll only twice — once in, once out.

All three active strategies are charged **the same per-side cost that Phase 8 calibrated**, read from
Phase 8's saved summary rather than re-typed here — one source of truth, so no variant can quietly
enjoy cheaper trading.

In [ ]:
cost = pd.read_csv('data/model_input/cost_summary.csv', index_col=0).iloc[:, 0]
BPS = float(cost['cost_bps'])

vae_net = pd.read_csv('data/model_input/wf_net_returns.csv', index_col=0, parse_dates=True)['ret']
vae_summary = pd.read_csv('data/model_input/wf_summary.csv')

g_net = g_ret - BPS / 1e4 * g_tvr
c_net = c_ret - BPS / 1e4 * c_tvr
bh_net = ret.mean(axis=1).loc[vae_net.index].copy()
bh_net.iloc[0] -= BPS / 1e4
bh_net.iloc[-1] -= BPS / 1e4

table = {}
for name, r, npairs in [('VAE', vae_net, vae_summary.n_pairs.mean()),
                        ('GARCH', g_net, g_summary.n_pairs.mean()),
                        ('Correlation', c_net, c_summary.n_pairs.mean()),
                        ('Buy&Hold', bh_net, np.nan)]:
    m, _ = perf(r)
    table[name] = {**m, 'avg_pairs': npairs}
table = pd.DataFrame(table).T
print(f'All strategies net of the same {BPS:.0f} bp/side toll:')
print(table.round(4).to_string())

## The head-to-head picture

One pot of £1 per runner, all net of the same costs, all over the same 27 quarters. **How to read
it:** the vertical order of the lines at the right edge is the ranking; the *smoothness* of each line
is the risk story (buy-and-hold usually ends highest in a bull decade but with far deeper dips — the
Sharpe column above, not the endpoint, is the risk-adjusted verdict).

**Either outcome is a result.** If the VAE line clearly beats GARCH and correlation, the learned
fingerprints add real selection skill. If it doesn't, the honest conclusion is that simple methods
suffice for this universe — a legitimate, publishable finding (the literature is short of credible
negative results, not long of them), and the walk-forward + cost framework built here is what makes
it credible either way.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
for name, r, colr in [('VAE (net)', vae_net, 'navy'), ('GARCH (net)', g_net, 'darkorange'),
                      ('Correlation (net)', c_net, 'seagreen'), ('Buy&Hold (net)', bh_net, 'grey')]:
    _, eq = perf(r)
    ax.plot(eq.index, eq.values, label=name, color=colr, lw=1.4 if name.startswith('VAE') else 1.0)
ax.axhline(1, color='k', lw=0.5)
ax.legend()
ax.set_title(f'Head-to-head, net of {BPS:.0f} bp/side: VAE vs GARCH vs Correlation vs Buy-and-Hold')
plt.tight_layout(); plt.show()

## Save the hand-off files

- **benchmark_returns.csv** — the four daily net return series side by side (Phase 10's overlay
  figure).
- **benchmark_metrics.csv** — the comparison table (the dissertation's head-to-head table).
- **garch_summary.csv / corr_summary.csv** — per-quarter attribution for the two challenger chains
  (supply, agreement, quarterly returns — the appendix material).

In [ ]:
pd.DataFrame({'vae': vae_net, 'garch': g_net, 'correlation': c_net, 'buyhold': bh_net}
             ).to_csv('data/model_input/benchmark_returns.csv')
table.to_csv('data/model_input/benchmark_metrics.csv')
g_summary.to_csv('data/model_input/garch_summary.csv', index=False)
c_summary.to_csv('data/model_input/corr_summary.csv', index=False)
print('Saved benchmark_returns / benchmark_metrics / garch_summary / corr_summary')

## Summary

Three challengers ran the **identical** quarterly walk-forward — same calendar, same tests, same
gates, same trading rule, same toll — with only the pair-finding step swapped: GARCH volatility
personalities, plain correlation, and buy-and-hold. The comparison table and overlay above are the
dissertation's closing argument: they isolate exactly what the VAE's learned fingerprints contribute,
positive or negative, with nothing else allowed to differ.